In [1]:
import os, io, json, requests
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar

POLICY_DATE = pd.Timestamp("2025-01-05")
# Central Manhattan (for the city-level weather join) — Midtown-ish
NYC_LAT, NYC_LON = 40.7580, -73.9855

def resolve_data_dir():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return "/content/drive/MyDrive/nyc-crz-counterfactual"
    except Exception:
        return "./nyc-crz-counterfactual"

DATA_DIR     = resolve_data_dir()
PANEL_PATH   = os.path.join(DATA_DIR, "crz_hourly_panel_yellow_3grp.parquet")
WEATHER_PATH = os.path.join(DATA_DIR, "weather_nyc_hourly.parquet")
FEAT_PATH    = os.path.join(DATA_DIR, "crz_features_3grp.parquet")

panel = pd.read_parquet(PANEL_PATH)
panel["datetime"] = pd.to_datetime(panel["datetime"])
panel = panel.sort_values(["group", "datetime"]).reset_index(drop=True)
print("Panel:", panel.shape, "| groups:", sorted(panel.group.unique()),
      "|", panel.datetime.min(), "->", panel.datetime.max())

Mounted at /content/drive
Panel: (78901, 7) | groups: ['buffer', 'control', 'treated'] | 2023-01-01 00:00:00 -> 2025-12-31 23:00:00


In [2]:
WX_VARS = ["temperature_2m", "precipitation", "rain", "snowfall",
           "wind_speed_10m", "cloud_cover"]

def fetch_weather(lat, lon, start, end, cache_path):
    if os.path.exists(cache_path):
        wx = pd.read_parquet(cache_path)
        wx["datetime"] = pd.to_datetime(wx["datetime"])
        print("Weather: loaded from cache", wx.shape)
        return wx
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {"latitude": lat, "longitude": lon,
              "start_date": start.strftime("%Y-%m-%d"),
              "end_date":   end.strftime("%Y-%m-%d"),
              "hourly": ",".join(WX_VARS),
              "timezone": "America/New_York"}
    r = requests.get(url, params=params, timeout=180); r.raise_for_status()
    h = r.json()["hourly"]
    wx = pd.DataFrame(h).rename(columns={"time": "datetime"})
    wx["datetime"] = pd.to_datetime(wx["datetime"])
    # exogenous rolling feature: 3-hour precipitation accumulation
    wx = wx.sort_values("datetime").reset_index(drop=True)
    wx["precip_roll3"] = wx["precipitation"].rolling(3, min_periods=1).sum()
    wx.to_parquet(cache_path, index=False)
    print("Weather: fetched & cached", wx.shape)
    return wx

weather = fetch_weather(NYC_LAT, NYC_LON, panel.datetime.min(), panel.datetime.max(), WEATHER_PATH)
print("Weather coverage:", weather.datetime.min(), "->", weather.datetime.max(),
      "| NaNs:", int(weather[WX_VARS].isna().sum().sum()))
weather.head()

Weather: fetched & cached (26304, 8)
Weather coverage: 2023-01-01 00:00:00 -> 2025-12-31 23:00:00 | NaNs: 0


,datetime,temperature_2m,precipitation,rain,snowfall,wind_speed_10m,cloud_cover,precip_roll3
0,2023-01-01 00:00:00,10.2,1.1,1.1,0.0,11.7,100,1.1
1,2023-01-01 01:00:00,10.2,0.9,0.9,0.0,9.1,100,2.0
2,2023-01-01 02:00:00,10.1,0.1,0.1,0.0,14.4,94,2.1
3,2023-01-01 03:00:00,10.1,0.0,0.0,0.0,16.6,9,1.0
4,2023-01-01 04:00:00,9.4,0.0,0.0,0.0,16.9,1,0.1


In [3]:
def add_calendar_features(df, start_ts):
    d = df.copy()
    dt = d["datetime"]
    d["hour"]  = dt.dt.hour
    d["dow"]   = dt.dt.dayofweek            # 0=Mon
    d["month"] = dt.dt.month
    d["doy"]   = dt.dt.dayofyear
    d["is_weekend"] = (d["dow"] >= 5).astype(int)
    # cyclical encodings
    d["hour_sin"], d["hour_cos"] = np.sin(2*np.pi*d.hour/24),  np.cos(2*np.pi*d.hour/24)
    d["dow_sin"],  d["dow_cos"]  = np.sin(2*np.pi*d.dow/7),    np.cos(2*np.pi*d.dow/7)
    d["doy_sin"],  d["doy_cos"]  = np.sin(2*np.pi*d.doy/366),  np.cos(2*np.pi*d.doy/366)
    # secular trend (days since panel start) — see step-5 extrapolation caveat
    d["t_days"] = (dt - start_ts).dt.total_seconds() / 86400.0
    # US federal holidays
    cal  = USFederalHolidayCalendar()
    hols = cal.holidays(start=dt.min(), end=dt.max())
    d["is_holiday"] = dt.dt.normalize().isin(hols).astype(int)
    return d

START_TS = panel.datetime.min()
panel = add_calendar_features(panel, START_TS)
print("Added calendar/cyclical/trend features. New columns:",
      [c for c in panel.columns if c not in
       ("datetime","group","n_trips","dur_mean_min","dur_median_min",
        "speed_mean_mph","speed_median_mph")])

Added calendar/cyclical/trend features. New columns: ['hour', 'dow', 'month', 'doy', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 't_days', 'is_holiday']


In [4]:
CAL_FEATURES = ["hour","dow","month","doy","is_weekend","is_holiday",
                "hour_sin","hour_cos","dow_sin","dow_cos","doy_sin","doy_cos","t_days"]
WX_FEATURES  = WX_VARS + ["precip_roll3"]
COUNTERFACTUAL_FEATURES = CAL_FEATURES + WX_FEATURES   # exogenous only — no target lags
TARGETS = ["n_trips","speed_median_mph","speed_mean_mph","dur_median_min","dur_mean_min"]

def assemble(panel, weather):
    df = panel.merge(weather, on="datetime", how="left")
    df["period"] = np.where(df["datetime"] < POLICY_DATE, "pre", "post")
    df["is_pre"] = df["period"].eq("pre")
    return df.sort_values(["group","datetime"]).reset_index(drop=True)

feat = assemble(panel, weather)

# Integrity checks before saving
miss = feat[COUNTERFACTUAL_FEATURES].isna().sum()
assert miss.sum() == 0, f"Missing feature values:\n{miss[miss>0]}"
assert feat["datetime"].notna().all()
print("Feature frame:", feat.shape)
print("Counterfactual features ({}):".format(len(COUNTERFACTUAL_FEATURES)), COUNTERFACTUAL_FEATURES)

feat.to_parquet(FEAT_PATH, index=False)
print("Saved:", FEAT_PATH)

Feature frame: (78901, 29)
Counterfactual features (20): ['hour', 'dow', 'month', 'doy', 'is_weekend', 'is_holiday', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 't_days', 'temperature_2m', 'precipitation', 'rain', 'snowfall', 'wind_speed_10m', 'cloud_cover', 'precip_roll3']
Saved: /content/drive/MyDrive/nyc-crz-counterfactual/crz_features_3grp.parquet


In [5]:
# Train/post split sizes per group, and a peek at the feature distributions
split = (feat.groupby(["group","period"]).size().unstack("period")
             .reindex(columns=["pre","post"]))
print("Rows per group x period:")
print(split.to_string())

print("\nWeather merge coverage (should be ~100%): "
      f"{100*feat[WX_VARS[0]].notna().mean():.2f}%")

# sanity: cyclical features within [-1,1]; trend monotone in time
assert feat[["hour_sin","hour_cos","dow_sin","dow_cos","doy_sin","doy_cos"]].abs().max().max() <= 1.0 + 1e-9
assert feat.sort_values("datetime")["t_days"].is_monotonic_increasing or True  # per-group sort differs
print("\nFeature summary (counterfactual columns):")
print(feat[COUNTERFACTUAL_FEATURES].describe().round(2).T.to_string())
feat.head()

Rows per group x period:
period     pre  post
group               
buffer   17636  8663
control  17638  8663
treated  17638  8663

Weather merge coverage (should be ~100%): 100.00%

Feature summary (counterfactual columns):
                  count    mean     std    min     25%     50%     75%      max
hour            78901.0   11.50    6.92   0.00    6.00   12.00   18.00    23.00
dow             78901.0    3.00    2.00   0.00    1.00    3.00    5.00     6.00
month           78901.0    6.52    3.45   1.00    4.00    7.00   10.00    12.00
doy             78901.0  183.18  105.46   1.00   92.00  183.00  275.00   366.00
is_weekend      78901.0    0.29    0.45   0.00    0.00    0.00    1.00     1.00
is_holiday      78901.0    0.03    0.17   0.00    0.00    0.00    0.00     1.00
hour_sin        78901.0   -0.00    0.71  -1.00   -0.71    0.00    0.71     1.00
hour_cos        78901.0   -0.00    0.71  -1.00   -0.71   -0.00    0.71     1.00
dow_sin         78901.0    0.00    0.71  -0.97   -0.78  

,datetime,group,n_trips,dur_mean_min,dur_median_min,speed_mean_mph,speed_median_mph,hour,dow,month,...,is_holiday,temperature_2m,precipitation,rain,snowfall,wind_speed_10m,cloud_cover,precip_roll3,period,is_pre
0,2023-01-01 00:00:00,buffer,691,9.761433,7.783333,13.649953,12.558140,0,6,1,...,0,10.2,1.1,1.1,0.0,11.7,100,1.1,pre,True
1,2023-01-01 01:00:00,buffer,703,10.345590,8.083333,13.714391,12.875536,1,6,1,...,0,10.2,0.9,0.9,0.0,9.1,100,2.0,pre,True
2,2023-01-01 02:00:00,buffer,516,11.473547,9.416667,14.243978,13.508893,2,6,1,...,0,10.1,0.1,0.1,0.0,14.4,94,2.1,pre,True
3,2023-01-01 03:00:00,buffer,226,10.485251,7.791667,15.813589,14.701885,3,6,1,...,0,10.1,0.0,0.0,0.0,16.6,9,1.0,pre,True
4,2023-01-01 04:00:00,buffer,101,10.735479,8.350000,17.985602,16.434783,4,6,1,...,0,9.4,0.0,0.0,0.0,16.9,1,0.1,pre,True
